# Muriel data recipes

Data calls only (no language model): about a second each, 1 credit per 1,000 records. The last cell uses the agent for a written recommendation, which is the slow step.

Set `MURIEL_API_KEY` before starting Jupyter. Violation records are not in the API yet.

Docs: https://docs.murielai.io (Traditional API → Data Recipes)

## Setup

Imports and one helper used by the recipes below: a system's full history, newest first, fetched 1,000 records at a time (the API pages up to 10,000).

In [ ]:
import os
import pandas as pd
from muriel import get_results, get_systems

# Reads MURIEL_API_KEY from the environment, or pass api_key="..." to each call.
assert os.environ.get("MURIEL_API_KEY"), "Set MURIEL_API_KEY first"

def system_history(pwsid, max_records=10_000, **filters):
    """Every result for a system, newest first, paging 1,000 at a time."""
    rows, offset = [], 0
    while offset < max_records:
        page = get_results(pwsid=pwsid, limit=1000, offset=offset, era="all", **filters)
        rows += page
        if len(page) < 1000:
            break
        offset += 1000
    return pd.DataFrame(rows)

## Everything on file for one system

A system's full record, then the most recent result for each contaminant.

In [ ]:
%%time
PWSID = "CA3010037"
df = system_history(PWSID)
print(f"{len(df):,} results for {df['system_name'].iloc[0] if len(df) else PWSID}")

latest = (df.sort_values("sample_collection_date")
            .groupby("result_analyte_name").tail(1)
            .sort_values("result_analyte_name"))
latest[["result_analyte_name", "sample_collection_date", "result_value", "result_unit", "result_non_detect"]]

## Lead over the action level across a state

Lead results above 15 µg/L (0.015 mg/L) in New Jersey, ranked by system. min_concentration is in mg/L.

In [ ]:
%%time
lead = pd.DataFrame(get_results(state="NJ", analyte="Lead",
                                detections_only=True, min_concentration=0.015, limit=500))
print(f"{len(lead):,} results")
(lead.groupby(["pwsid", "system_name"]).size()
     .sort_values(ascending=False).head(15).rename("samples over 15 µg/L"))

## PFAS detections in a state, by compound

Detections in the pfas analyte group in California: how many, in how many systems, and the latest.

In [ ]:
%%time
pfas = pd.DataFrame(get_results(state="CA", analyte_group="pfas",
                                detections_only=True, limit=1000))
print(f"{len(pfas):,} detections")
(pfas.groupby("result_analyte_name")
     .agg(detections=("pwsid", "size"), systems=("pwsid", "nunique"),
          latest=("sample_collection_date", "max"))
     .sort_values("detections", ascending=False).head(10))

## The largest systems in a state

Every system in New Jersey (limit=None pages for you), largest first, with how many results Muriel holds and the newest sample.

In [ ]:
%%time
systems = pd.DataFrame(get_systems(state="NJ", limit=None)["records"])
print(f"{len(systems):,} systems")
systems.sort_values("population_served_count", ascending=False)[
    ["pwsid", "pws_name", "population_served_count", "results_held", "latest_sample"]].head(10)

## A due-diligence screen without the agent

The agent's due-diligence example, built from data calls in seconds. It flags results over a federal limit (critical when recent) and key contaminants with nothing on file. The limits and rules are illustrative: one result over a limit is not a violation, since compliance is judged on running averages and by the state, so treat this as a screen. concentration_value is returned in mg/L.

In [ ]:
%%time
from datetime import date, timedelta

# Federal limits in mg/L (lead and copper are action levels, not MCLs).
LIMITS_MG_L = {
    "ARSENIC": 0.010, "NITRATE": 10.0, "NITRITE": 1.0, "FLUORIDE": 4.0, "BARIUM": 2.0,
    "PFOA": 4e-6, "PFOS": 4e-6, "LEAD": 0.015, "COPPER": 1.3,
    "TOTAL TRIHALOMETHANES": 0.080, "TTHM": 0.080, "TOTAL THMS": 0.080,
    "TOTAL HALOACETIC ACIDS (HAA5)": 0.060, "HAA5": 0.060,
}
TO_MG_L = {"MG/L": 1, "PPM": 1, "UG/L": 1e-3, "µG/L": 1e-3, "PPB": 1e-3, "NG/L": 1e-6, "PPT": 1e-6}
KEY_CONTAMINANTS = ("LEAD", "ARSENIC", "NITRATE", "PFOA")

def to_mg_l(row):
    value = row.get("concentration_value")
    unit = str(row.get("concentration_units") or row.get("result_unit") or "").upper().replace(" ", "")
    factor = TO_MG_L.get(unit)
    return None if value is None or factor is None else float(value) * factor

def show(mg_l):
    """A concentration in the unit it is usually quoted in."""
    if mg_l < 1e-4:
        return f"{mg_l * 1e6:.3g} ng/L"
    if mg_l < 0.1:
        return f"{mg_l * 1e3:.3g} µg/L"
    return f"{mg_l:.3g} mg/L"

def due_diligence(pwsid, recent_years=3):
    results = system_history(pwsid)
    cutoff = (date.today() - timedelta(days=365 * recent_years)).isoformat()
    flags, held = [], set()
    if len(results):
        r = results.copy()
        r["analyte"] = r["result_analyte_name"].str.upper().str.strip()
        r["limit_mg_l"] = r["analyte"].map(LIMITS_MG_L)
        r["mg_l"] = r.apply(to_mg_l, axis=1)
        held = set(r["analyte"])
        over = r[r["limit_mg_l"].notna() & r["mg_l"].notna() & (r["mg_l"] > r["limit_mg_l"])]
        for analyte, g in over.groupby("analyte"):
            last = str(g["sample_collection_date"].max())
            flags.append({"issue_type": "result over federal limit", "analyte": analyte,
                          "severity": "critical" if last >= cutoff else "moderate",
                          "description": f"{len(g)} result(s) over {show(g['limit_mg_l'].iloc[0])}, "
                                         f"highest {show(g['mg_l'].max())}, most recent {last}",
                          "date_identified": last})
    for name in KEY_CONTAMINANTS:
        if name not in held:
            flags.append({"issue_type": "no results on file", "analyte": name, "severity": "minor",
                          "description": "nothing in the published record", "date_identified": None})
    critical = sum(f["severity"] == "critical" for f in flags)
    assessment = "fail" if critical else "caution" if flags else "pass"
    name = results["system_name"].iloc[0] if len(results) else pwsid
    return {"pwsid": pwsid, "system_name": name, "results_on_file": len(results),
            "flags": flags, "critical_count": critical, "overall_assessment": assessment}

report = due_diligence("CA3010037")
status_emoji = {"pass": "✅", "caution": "⚠️", "fail": "❌"}
print(f"{status_emoji[report['overall_assessment']]} {report['system_name']} "
      f"({report['results_on_file']:,} results on file)")
print(f"Assessment: {report['overall_assessment'].upper()}, {report['critical_count']} critical")
for f in report["flags"]:
    print(f"  [{f['severity'].upper()}] {f['issue_type']}: {f['analyte']} {f['description']}")

## The agent for the narrative only

If you want a written recommendation, ask the agent for just that, scoped to the system. This is the slow step (tens of seconds); the recipes above take seconds.

In [ ]:
%%time
from muriel import Agent

with Agent() as agent:
    answer = agent.query(
        "In two or three sentences, give a due-diligence recommendation for this system "
        "based on its results over federal limits and any monitoring gaps.",
        pwsid="CA3010037",
    )
print(answer.summary)